# Lab 2: Distributions

> **Course:** Data Science Foundations
> **Lesson:** Day 2 Lab, companion to `2-distributions.md`
> **Dataset:** `taxis` (seaborn). 6,433 New York taxi rides from March 2019, with pickup time, distance, fare, tip, and borough
> **Estimated time:** 90 minutes

## Lab Objectives

- Identify an unknown distribution from its histogram and summary statistics
- Use `scipy.stats` to draw PMFs and PDFs and check that probabilities sum (or integrate) to 1
- Simulate **Bernoulli** and **Binomial** data, the distributions behind Thursday's A/B tests
- Compute **z-scores** and test the 68-95-99.7 rule on real, skewed data
- Read a **QQ plot** to judge Normality
- Look for a Poisson process in real ride counts, and explain overdispersion

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

taxis = sns.load_dataset("taxis")
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
print(f"Shape: {taxis.shape}")
taxis.head()

## Part 1: Mystery Distributions

The cell below generates four samples, **A to D**, and shuffles which letter gets which distribution. Don't try to decode the generator. Identify each sample from the data alone.

In [ ]:
_gen = np.random.default_rng(2024)
_samples = [
    _gen.normal(50, 8, 3000),
    _gen.uniform(0, 60, 3000),
    _gen.poisson(3, 3000).astype(float),
    _gen.exponential(10, 3000),
]
_order = _gen.permutation(4)
mystery = {letter: _samples[i] for letter, i in zip("ABCD", _order)}

In [ ]:
# TODO 1.1: a 2 x 2 grid of histograms, one per mystery sample, titled with its letter.
# Tip: for a sample that contains only whole numbers, use discrete=True.

In [ ]:
# TODO 1.2: one row per sample with: mean, var, var_to_mean (var / mean), skew, min, max, all_integers
profile = None
profile

**Question 1.** Name the distribution behind each letter and give two pieces of evidence for each. One of them is none of Uniform, Poisson, or Normal. Describe its shape and suggest what it might model.

**Your answer:**

*Write your answer here.*

## Part 2: PMFs, PDFs, and Parameters

`scipy.stats` gives every distribution the same interface: `.pmf` (discrete) or `.pdf` (continuous), `.cdf`, `.ppf`, `.mean()`, `.var()`, `.rvs()`.

In [ ]:
# TODO 2.1: on ONE axes, plot the Poisson PMF for lambda in [1, 4, 10, 30], for k = 0..50 (lines with markers)

# TODO 2.2: on a second axes, plot Normal PDFs for (mu, sigma) = (0, 1), (0, 2), (3, 1) over x in [-8, 8]

In [ ]:
# TODO 2.3: check that total probability is 1 in each case
total_poisson = None   # sum of Poisson(4) PMF over k = 0..100
total_normal = None    # area under Normal(0, 2) PDF: use np.trapezoid(pdf_values, x) on a fine grid over [-20, 20]
total_uniform = None   # area under Uniform on [2, 10] PDF (stats.uniform(loc=2, scale=8)) over [0, 12]
print(total_poisson, total_normal, total_uniform)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for total in (total_poisson, total_normal, total_uniform):
    assert abs(total - 1) < 1e-3
print("passed")

**Question 2.** The Uniform(2, 10) PDF has height 0.125 everywhere on [2, 10]. A PDF of a Normal with σ = 0.1 peaks at about 4. How can a density be greater than 1 when probabilities cannot be?

**Your answer:**

*Write your answer here.*

## Part 3: Bernoulli and Binomial (extension)

The lesson covered Uniform, Poisson, and Normal. Thursday's A/B tests use two more:

- **Bernoulli($p$):** a single yes/no trial (did this visitor sign up?). The outcome is 1 with probability $p$, else 0.
- **Binomial($n$, $p$):** the number of successes in $n$ independent Bernoulli trials. Mean $np$, variance $np(1-p)$.

In [ ]:
n_visitors, p_convert, n_days = 500, 0.08, 2000

# TODO 3.1: simulate n_days days. Each day, n_visitors visitors each convert with probability p_convert.
# Build the Bernoulli outcomes explicitly (a 0/1 array of shape (n_days, n_visitors)), then count per day.
daily_conversions = None

# TODO 3.2: compare the observed mean and variance with n*p and n*p*(1-p)

# TODO 3.3: histogram of daily_conversions (discrete=True, stat="probability")
# overlaid with stats.binom(n_visitors, p_convert).pmf as red dots

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert len(daily_conversions) == n_days
assert abs(daily_conversions.mean() - 40) < 1
print("passed")

**Question 3.** The Binomial histogram looks almost Normal. Why? How does that help when comparing two conversion rates on Thursday?

**Your answer:**

*Write your answer here.*

## Part 4: z-Scores and the Empirical Rule on Real Data

A **z-score** says how many standard deviations a value lies from the mean:

$$z = \frac{x - \mu}{\sigma}$$

The 68-95-99.7 rule is a statement about z-scores: for Normal data, about 68% have $|z| \le 1$, 95% have $|z| \le 2$, and 99.7% have $|z| \le 3$.

In [ ]:
def zscores(values):
    # TODO: return (values - mean) / std  (use ddof=1)
    return None


def empirical_rule_check(values):
    # TODO: return {1: share with |z| <= 1, 2: share with |z| <= 2, 3: share with |z| <= 3}
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
z = zscores([2, 4, 6])
assert np.allclose(z, [-1, 0, 1])
print("passed")

In [ ]:
# TODO 4.2: a table with one row per dataset (fare, log(fare), simulated Normal with 6433 values)
# and columns within_1sd, within_2sd, within_3sd, skew. Add a final row 'Normal theory' = 0.683, 0.954, 0.997.
rule_table = None
rule_table

In [ ]:
# TODO 4.3: which ride has the most extreme distance z-score? Show pickup, distance, fare and its z-score.

**Question 4.** Does the empirical rule hold for `fare`? For `log(fare)`? What goes wrong if you flag "|z| > 3" as anomalies on skewed data?

**Your answer:**

*Write your answer here.*

## Part 5: QQ Plots (extension)

The lesson warned that a bell-shaped histogram is not proof of Normality and promised a better tool. A **QQ plot** sorts your data and plots it against the values a perfect Normal would produce at the same quantiles. **Normal data falls on the straight line.** A curve bending upward on the right means a heavy right tail.

`stats.probplot(values, dist="norm", plot=ax)` draws one.

In [ ]:
# TODO 5.1: 1 x 3 QQ plots for: simulated Normal, fare, log(fare). Title each panel.

**Question 5.** Describe the shape of each QQ plot and what it tells you.

**Your answer:**

*Write your answer here.*

## Part 6: Looking for a Poisson Process

The number of pickups per hour sounds like a textbook Poisson variable: independent events, counted in fixed intervals. Test that idea.

In [ ]:
# TODO 6.1: count rides in every calendar hour from the first to the last pickup.
# Hours with no rides must appear with a count of 0 (hint: pd.date_range(..., freq="h") and .reindex(fill_value=0))
rides_per_hour = None

# TODO 6.2: print the mean, variance and variance/mean ratio of rides_per_hour

In [ ]:
# TODO 6.3: plot the average rides per hour-of-day (0-23). Is the rate constant through the day?

# TODO 6.4: keep only the counts for one hour of the day (e.g. 18:00 to 18:59), then repeat 6.2.
# Plot their histogram (stat="probability", discrete=True) with a Poisson PMF using lambda = their mean.

**Question 6.** Are all-day hourly counts Poisson? Why is the variance so much larger than the mean? What changed when you fixed the hour of day?

**Your answer:**

*Write your answer here.*

## Part 7: Wrap-Up

In 3 to 4 sentences: which distribution would you use to model (a) the fare of a ride, (b) whether a rider tips, (c) the number of rides in the 18:00 hour, and why?

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.